In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

In [3]:
healthexp = sns.load_dataset('healthexp')
healthexp.head(100)

,Year,Country,Spending_USD,Life_Expectancy
0,1970,Germany,252.311,70.6
1,1970,France,192.143,72.2
2,1970,Great Britain,123.993,71.9
3,1970,Japan,150.437,72.0
4,1970,USA,326.961,70.9
...,...,...,...,...
95,1991,Canada,1805.209,77.6
96,1991,France,1558.033,77.2
97,1991,Great Britain,842.797,75.9
98,1991,Japan,1166.430,79.1


In [4]:
healthexp = pd.get_dummies(healthexp)


In [5]:
X = healthexp.drop(['Life_Expectancy'], axis=1)

In [6]:
y = healthexp['Life_Expectancy']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=19)


In [8]:
rfr = RandomForestRegressor(random_state=13)


In [9]:
rfr.fit(X_train, y_train)


RandomForestRegressor(random_state=13)

In [10]:
y_pred = rfr.predict(X_test)


In [11]:
mean_absolute_error(y_test, y_pred)


0.25916363636361917

In [12]:
mean_squared_error(y_test, y_pred)


0.10221141818181627

In [13]:
r2_score(y_test, y_pred)


0.9910457602615238

**Optuna**

In [14]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 32)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf
    )

    score = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    return score.mean()


In [15]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler(seed=42)) # Default is random Search|

[I 2025-08-15 13:48:51,395] A new study created in memory with name: no-name-5dddd57a-08fb-4bcb-a20d-bd41b6d0a018


In [16]:
study.optimize(objective, n_trials=300)

[I 2025-08-15 13:48:53,908] Trial 0 finished with value: -2.540931874729219 and parameters: {'n_estimators': 144, 'max_depth': 48, 'min_samples_split': 24, 'min_samples_leaf': 20}. Best is trial 0 with value: -2.540931874729219.
[I 2025-08-15 13:48:54,221] Trial 1 finished with value: -3.1510909388980486 and parameters: {'n_estimators': 89, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 28}. Best is trial 0 with value: -2.540931874729219.
[I 2025-08-15 13:48:54,887] Trial 2 finished with value: -3.7618662791144053 and parameters: {'n_estimators': 200, 'max_depth': 39, 'min_samples_split': 2, 'min_samples_leaf': 32}. Best is trial 0 with value: -2.540931874729219.
[I 2025-08-15 13:48:55,796] Trial 3 finished with value: -0.8803599985779715 and parameters: {'n_estimators': 258, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 6}. Best is trial 3 with value: -0.8803599985779715.
[I 2025-08-15 13:48:56,249] Trial 4 finished with value: -1.6607141608073597 and para

In [17]:
print(study.best_params)
print(study.best_value)

{'n_estimators': 284, 'max_depth': 45, 'min_samples_split': 3, 'min_samples_leaf': 1}
-0.1976662730088205


In [18]:
best_params = study.best_params

In [19]:
optuna.visualization.plot_optimization_history(study)

In [20]:
optuna.visualization.plot_parallel_coordinate(study)

In [21]:
optuna.visualization.plot_slice(study, params=['n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split'])


In [22]:
optuna.visualization.plot_param_importances(study)

In [23]:
best_n_estimators = best_params['n_estimators']
best_max_depth = best_params['max_depth']
best_min_samples_split = best_params['min_samples_split']
best_min_samples_leaf = best_params['min_samples_leaf']

In [24]:
best_model = RandomForestRegressor(n_estimators=best_n_estimators,
max_depth=best_max_depth,
min_samples_split=best_min_samples_split,
min_samples_leaf=best_min_samples_leaf)
best_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=45, min_samples_split=3, n_estimators=284)

In [25]:
y_pred = best_model.predict(X_test)

In [26]:
mean_absolute_error(y_test, y_pred)

0.25453941324718854

In [27]:
mean_squared_error(y_test, y_pred)

0.09721359994452476

In [28]:
r2_score(y_test, y_pred)

0.9914835945413146